In [20]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from tqdm import tqdm
from sklearn.cluster import DBSCAN

In [21]:
milk = pd.read_csv(r"D:\Machine_Learning_CDAC-AI\a] Datasets\milk.csv", index_col=0, header=3)
milk

,88.4,2.2,2.7,6.4,0.18
MONKEY,,,,,
DONKEY,90.3,1.7,1.4,6.2,0.40
HIPPO,90.4,0.6,4.5,4.4,0.10
CAMEL,87.7,3.5,3.4,4.8,0.71
BISON,86.9,4.8,1.7,5.7,0.90
BUFFALO,82.1,5.9,7.9,4.7,0.78
GUINEA PIG,81.9,7.4,7.2,2.7,0.85
CAT,81.6,10.1,6.3,4.4,0.75
FOX,81.6,6.6,5.9,4.9,0.93
LLAMA,86.5,3.9,3.2,5.6,0.80


In [23]:
milk = pd.read_csv(r"D:\Machine_Learning_CDAC-AI\a] Datasets\milk.csv", index_col=0, header=3)
milk.columns = ['water', 'protein', 'fat', 'lactose', 'ash']
milk_clean = milk[features_to_scale]

In [24]:
features_to_scale = ['water', 'protein', 'fat', 'lactose', 'ash']
milk_clean = milk[features_to_scale]

# 3. Now scale the data safely (this defines 'df_scaled')
scaler = StandardScaler().set_output(transform='pandas')
df_scaled = scaler.fit_transform(milk_clean)

# 4. Run DBSCAN now that 'df_scaled' exists!
dbscan = DBSCAN(eps=0.4, min_samples=2)
dbscan.fit(df_scaled)

# 5. Copy and view results
df_copy = milk.copy()
df_copy['Cluster'] = dbscan.labels_
df_copy

,water,protein,fat,lactose,ash,Cluster
MONKEY,,,,,,
DONKEY,90.3,1.7,1.4,6.2,0.40,-1
HIPPO,90.4,0.6,4.5,4.4,0.10,-1
CAMEL,87.7,3.5,3.4,4.8,0.71,0
BISON,86.9,4.8,1.7,5.7,0.90,1
BUFFALO,82.1,5.9,7.9,4.7,0.78,2
GUINEA PIG,81.9,7.4,7.2,2.7,0.85,-1
CAT,81.6,10.1,6.3,4.4,0.75,-1
FOX,81.6,6.6,5.9,4.9,0.93,2
LLAMA,86.5,3.9,3.2,5.6,0.80,1


In [25]:
silhouette_score(df_scaled, dbscan.labels_)

-0.08298768056390536

In [26]:
df_scaled_inliner = df_scaled.copy()
df_scaled_inliner['Cluster']=dbscan.labels_
df_scaled_inliner = df_scaled_inliner[df_scaled_inliner["Cluster"] != -1]
df_scaled_inliner

,water,protein,fat,lactose,ash,Cluster
MONKEY,,,,,,
CAMEL,0.866139,-0.951747,-0.758556,0.584510,-0.502579,0
BISON,0.803095,-0.574216,-0.920025,1.120311,-0.097804,1
BUFFALO,0.424831,-0.254767,-0.331140,0.524977,-0.353452,2
FOX,0.385428,-0.051481,-0.521103,0.644044,-0.033893,2
LLAMA,0.771573,-0.835583,-0.777553,1.060778,-0.310844,1
ZEBRA,0.747932,-1.096951,-0.625582,0.882177,-0.523883,0
SHEEP,0.416950,-0.341890,-0.473612,0.524977,-0.076501,2
DEER,-0.851811,1.052069,0.789641,-0.725225,0.967393,3
REINDEER,-0.938497,1.139192,0.846630,-0.784759,0.967393,3


In [27]:
len(df_scaled_inliner['Cluster'].unique())

4

In [28]:
silhouette_score(df_scaled_inliner.drop('Cluster', axis=1), df_scaled_inliner['Cluster'])

0.5873520086331863

## Trying Different Combinations of `eps` and `min_samples`

In [29]:
eps = np.linspace(0.01,2, 100)
min_samp = [2,3,4,5]
scores = []
for e in eps:
    for m in min_samp:
        dbscan = DBSCAN(eps = e, min_samples = m)
        dbscan.fit(df_scaled)
        
        df_scaled_inliner = df_scaled.copy()
        df_scaled_inliner['Cluster']=dbscan.labels_
        df_scaled_inliner = df_scaled_inliner[df_scaled_inliner["Cluster"] != -1]
        
        if  len(df_scaled_inliner['Cluster'].unique()) >= 2:
            scores.append([e, m, silhouette_score(df_scaled_inliner.drop('Cluster', axis=1), 
                                                  df_scaled_inliner['Cluster'])])

df_scores = pd.DataFrame(scores, columns=['epsilon', 'sample', 'scores'])
df_scores.sort_values('scores', ascending=False)

,epsilon,sample,scores
0,0.331616,2,0.902458
1,0.351717,2,0.902458
2,0.371818,2,0.902458
53,0.874343,3,0.674169
61,0.954747,3,0.674169
...,...,...,...
62,0.974848,2,0.449384
64,0.994949,2,0.449384
60,0.954747,2,0.449384
56,0.914545,2,0.449384


In [30]:
dbscan = DBSCAN(eps = 0.311515, min_samples=2)
dbscan.fit(df_scaled)
dbscan.labels_   # -1 are too much try diff values

array([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        0,  0, -1, -1, -1])

In [31]:
df_scores.iloc[15:25,:]

,epsilon,sample,scores
15,0.512525,3,0.531746
16,0.532626,2,0.603357
17,0.532626,3,0.531746
18,0.552727,2,0.603357
19,0.552727,3,0.531746
20,0.572828,2,0.603357
21,0.572828,3,0.531746
22,0.592929,2,0.603357
23,0.592929,3,0.531746
24,0.613030,2,0.603357


In [32]:
dbscan = DBSCAN(eps = 0.432121, min_samples=3)
dbscan.fit(df_scaled)
dbscan.labels_ 

array([-1, -1,  0,  0,  1, -1, -1,  1,  0, -1, -1,  0,  1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1])

## Using ipython Widgets

In [33]:
from ipywidgets import widgets, interact

In [34]:
def dbscan_clust(e,m):
    dbscan = DBSCAN(eps = e, min_samples=m)
    dbscan.fit(df_scaled)
    return dbscan.labels_ 
    

In [35]:
interact(dbscan_clust, e = widgets.FloatSlider(min = 0.01, max = 2, step = 0.01, value = 0.3),
                        m = widgets.IntSlider(min = 2, max = 5, step =1, value = 2))

interactive(children=(FloatSlider(value=0.3, description='e', max=2.0, min=0.01, step=0.01), IntSlider(value=2…

<function __main__.dbscan_clust(e, m)>

In [36]:
interact(dbscan_clust, e = widgets.FloatSlider(min = 0.01, max = 2, step = 0.01, value = 0.3),
                        m = [2,3,4,5])

interactive(children=(FloatSlider(value=0.3, description='e', max=2.0, min=0.01, step=0.01), Dropdown(descript…

<function __main__.dbscan_clust(e, m)>

In [37]:
# Using Manhattan

eps = np.linspace(0.01,2, 100)
min_samp = [2,3,4,5]
scores = []
for e in eps:
    for m in min_samp:
        dbscan = DBSCAN(eps = e, min_samples = m, metric='manhattan')
        dbscan.fit(df_scaled)
        
        df_scaled_inliner = df_scaled.copy()
        df_scaled_inliner['Cluster']=dbscan.labels_
        df_scaled_inliner = df_scaled_inliner[df_scaled_inliner["Cluster"] != -1]
        
        if  len(df_scaled_inliner['Cluster'].unique()) >= 2:
            scores.append([e, m, silhouette_score(df_scaled_inliner.drop('Cluster', axis=1), 
                                                  df_scaled_inliner['Cluster'])])

df_scores = pd.DataFrame(scores, columns=['epsilon', 'sample', 'scores'])
df_scores.sort_values('scores', ascending=False)

,epsilon,sample,scores
0,0.532626,2,0.902458
1,0.552727,2,0.902458
2,0.572828,2,0.902458
3,0.592929,2,0.902458
4,0.613030,2,0.902458
...,...,...,...
130,1.839192,2,0.405568
122,1.758788,2,0.405568
140,1.939697,2,0.405568
136,1.899495,2,0.405568
